# SHFQC QA Channel -- Raw Scope Capture

Captures raw ADC waveform data from an SHFQC QA channel's built-in **Scope** module,
independent of any qubit readout/integration pipeline.

**Can this be done with the LabOneQ architecture?** Partly. LabOneQ's `Experiment`/pulse-
sequencing DSL has no Scope-module concept -- `AcquisitionType.RAW` is a different thing
(raw ADC samples captured *through the acquire/integration pipeline*, tied to a defined
integration window inside a compiled experiment). The actual hardware Scope block
(`device.scopes[...]`) sits outside that pipeline entirely and isn't reachable from
`exp.acquire(...)`.

That said, `session.devices[...]` in LabOneQ returns the *exact same* `zhinst.toolkit`
device object toolkit code uses directly (confirmed from the real `laboneq` source --
`Session.devices` is typed as `ToolkitDevices`) -- so `session.devices["device_shfqc"].scopes[0]`
would work too, if you already have a LabOneQ session open for something else. Since raw
scope capture needs no pulse sequencing, logical signal lines, or qubit/QPU setup at all,
this notebook uses **plain `zhinst.toolkit`** directly -- simpler, no `DeviceSetup`
overhead for a task that isn't really an "experiment" in the LabOneQ sense.


In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from zhinst.toolkit import Session, SHFQAChannelMode

pio.renderers.default = "browser"

## 1. Connect

Reuses the same SHFQC (`dev12073`, `localhost:8004`) as the Amplitude Rabi Chevron
notebook in this folder.


In [2]:
session = Session("localhost")
device = session.connect_device("dev12073")

## 2. Configure the QA Channel

`center_frequency` is the QA channel's analog mixer/LO frequency (Hz) -- set this to
whatever RF band you want to capture. `input_range`/`output_range` are in dBm.


In [3]:
CHANNEL_INDEX = 0

device.qachannels[CHANNEL_INDEX].configure_channel(
    center_frequency=6.8e9,   # TODO: set to your target RF frequency (Hz)
    input_range=0,            # TODO: dBm -- set to a safe range for your signal level
    output_range=-5,          # dBm
    mode=SHFQAChannelMode.READOUT,
)
device.qachannels[CHANNEL_INDEX].input.on(1)
device.qachannels[CHANNEL_INDEX].output.on(1)


## 3. Configure & Run the Scope

`trigger_input=None` puts the scope in **self-triggered / free-run** mode -- captures
immediately with no external or sequencer trigger needed, since nothing else is driving
this QA channel. If you already have a pulse sequence running elsewhere and want the
capture synchronized to it instead, set `trigger_input=f"channel{CHANNEL_INDEX}_sequencer_monitor0"`
(requires an active AWG/sequencer program generating that trigger).

`num_samples` below is 10 us at the SHFQA's fixed 2 GSa/s scope sample rate.


In [4]:
SHFQA_SCOPE_SAMPLING_RATE = 2e9   # fixed, Hz
CAPTURE_DURATION = 10e-6          # s
NUM_SAMPLES = int(CAPTURE_DURATION * SHFQA_SCOPE_SAMPLING_RATE)

device.scopes[0].configure(
    input_select={0: f"channel{CHANNEL_INDEX}_signal_input"},
    num_samples=NUM_SAMPLES,
    trigger_input=None,   # self-triggered / free-run -- see note above
    num_segments=1,
    num_averages=1,
    trigger_delay=0.0,
)

device.scopes[0].run(single=True)
scope_data, *_ = device.scopes[0].read()
raw = scope_data[0]   # complex ADC samples (I + jQ), shape (num_segments, num_samples)
print("Captured shape:", raw.shape, "dtype:", raw.dtype)


Captured shape: (20000,) dtype: complex128


## 4. Plot the Captured Waveform


In [5]:
trace = np.atleast_2d(raw)[0]   # normalize in case the segment axis was squeezed away
t = np.arange(len(trace)) / SHFQA_SCOPE_SAMPLING_RATE * 1e6   # us

fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=trace.real, name="I", line=dict(color="steelblue", width=0.8)))
fig.add_trace(go.Scatter(x=t, y=trace.imag, name="Q", line=dict(color="darkorange", width=0.8)))
fig.update_xaxes(title_text="Time (us)")
fig.update_yaxes(title_text="Amplitude (a.u.)")
fig.update_layout(title="SHFQC QA Scope Capture", template="plotly_white",
                   paper_bgcolor="white", plot_bgcolor="white")
fig.show()
